# Abstract für Gleichstellungsplan

## Zentrales Element: Zuordnung von Geschlecht über "Gender-Guesser", Manuelle Nachbearbeitung für MPG-Angehörige zur Korrektur der automatisierten Zuordnungen und manuelle Anreicherung mit Positionen

In [1]:
# Zentrale Komponente: Gender-Guesser - Installation über pip nur inital nötig 
#!pip install gender-guesser

In [1]:
import gender_guesser.detector as gender
import pandas as pd
import numpy as np
import re
import pdb
from collections import defaultdict

In [5]:
# Defintion der Funktion für Gendererkennung, die an ansprechender Stelle aufgerufen wird

def get_gender(name):                        
  d = gender.Detector()
  return d.get_gender(name)

# Daten aus Openrefine

### Wichtig, dass die Daten zunächst in Openrefine eingelesen werden, da das JSON sonst für Python nicht verarbeitbar ist!   

Es macht ggf. Sinn, einen Gesamtextrakt zu ziehen, um dann per Skript die für den Bericht relevanten Daten zu holen?

In [2]:
# Einlesen der Date aus dem Notebook "PuRe Rohdatenextrakt.jpynb"
df_pure = pd.read_excel('Daten/df_pure.xlsx')

# Zur Sicherheit die Anzahl an IDs = Datensätzen zählen
df_pure['90999_ItemID'].nunique() 

447

In [3]:
# Reduzierung auf die für die Analyse relevanten Spalten 

df_glp = df_pure[['1002_Genre',
                  '1_Nachname',
                  '2_Vorname',
                  'Name',
                  '4_Cone',
                  '5_Affiliation',
                  '1101_Quellentitel',
                  '1103_Ersterscheinungsdatum',
                  '90999_ItemID']] 

In [6]:
# Aufbereitung der Daten zur Gendererkennung (läuft über die Vornamen)

df_vorname = df_glp[['2_Vorname', 'Name', '4_Cone']]    # Nur die Spalten mit den Vornamen und Item-IDs für die Gendererkennung
df_vorname = df_vorname.drop_duplicates()           # Duplikate entfernen, damit jeder Vorname nur einmal vorkommt
df_vorname = df_vorname.dropna()                    # Entfernen aller Zeilen mit fehlenden Vornamen / Cone-IDs, da diese für die Gendererkennung nicht relevant sind

df_vorname['3_Gender'] = df_vorname['2_Vorname'].apply(get_gender) # Hier Aufruf der Gender-Erkennungsfunktion

In [7]:
# Erster Eindruck der Zuordnungen, zeigt auch Fehler oder Unsicherheiten an 

gender_counts = df_vorname['3_Gender'].value_counts()
gender_counts

3_Gender
male           55
female         41
unknown        24
mostly_male     1
andy            1
Name: count, dtype: int64

In [15]:
# Liste wird erzeugt zur Prüfung und manuellen Nachbereitung 

# Zuerst Zusammenführung der ursprünglichen Personendaten mit Genderinformationen
df_personen = df_glp[['Name', '4_Cone']]
df_personenliste_gendered = pd.merge(df_personen, df_vorname[['Name', '3_Gender']], on='Name', how='left')  

# Neue Liste, die für die manuelle Nachbarbeitung genutzt wird  
df_personenliste_gendered.dropna(subset=['Name'], inplace=True)  #Leerzeilen löschen 
df_personenliste_gendered.drop_duplicates(inplace=True)
df_personenliste_gendered['6_Status'] = None             #einfügen einer leeren Spalte für die Statusinfos

# Reduzierung auf die Personen mit Cone-Eintrag
only_cone = ['CONE']  
df_personenliste_cropped = df_personenliste_gendered[df_personenliste_gendered['4_Cone'].isin(only_cone)]
df_personenliste_cropped.to_excel('Daten/Pure_Personen_gendered.xlsx', index=False) # Export für Excel

# Bearbeitungsschritte der Datei "Pure_Personen_gendered.xlsx":  


1. Prüfung der korrekten Geschlechterzuordnung (auf alle Fälle für die Personen mit Cone-ID, also Institutszugehörigkeit)   
2. Hinzufügen des Status nach folgendem Schema

- Doctoral Student
- Postdoc
- Senior Researcher
- Research Group Leader
- Director
- Director emeritus
- Visiting Researcher
- Other

### Nach Fertigstellung Datei wieder hochladen!

In [4]:
# Achtung, dass der Name der Eingabedatei stimmt!
df_personenliste_gendered_korr = pd.read_excel('Daten/Pure_Personen_gendered.xlsx')

# Extrakt der relevanten Spalten
df_gender = df_personenliste_gendered_korr[['Name', '3_Gender','6_Status']]

# Datei in das Hauptdatenframe einspielen
merged_df = pd.merge(df_glp, df_gender[['Name', '3_Gender', '6_Status']], on='Name', how='left')

In [5]:
# Dublikate Entfernen
merged_df.drop_duplicates(inplace=True) 

#Kontrollfrage, ob es sich immer noch um die gleiche Anzahl Titel handelt
merged_df['90999_ItemID'].nunique() 

447

In [6]:
df_glp_cleaned = merged_df.rename(columns={
    '1002_Genre':'Genre',
    '1103_Ersterscheinungsdatum' : 'Datum',
    '1_Nachname':'Nachname',
    '2_Vorname' : 'Vorname',
    '3_Gender':'Gender',
    '4_Cone':'Cone',
    '6_Status' : 'Status',
    '5_Affiliation':'Affiliation',
    '90999_ItemID' : 'ItemID' }) 

df_glp_cleaned 

,Genre,Nachname,Vorname,Name,Cone,Affiliation,1101_Quellentitel,Datum,ItemID,Gender,Status
0,ARTICLE,Cigna,Luca,Luca Cigna,CONE,"Politische Ökonomie, MPI for the Study of Soci...",New Political Economy,2026.0,item_3699285,male,Postdoc
1,ARTICLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,item_3699285,NaN,NaN
3,ARTICLE,Lopez-Uroz,Nina,Nina Lopez-Uroz,CONE,Umstrittene Ökologien in kapitalistischen Demo...,NaN,NaN,item_3699285,female,Postdoc
6,ARTICLE,Gruber,Stephan,Stephan Gruber,CONE,"Wirtschaftssoziologie, MPI for the Study of So...",Global Perspectives,2026.0,item_3705148,male,Doctoral Student
7,ARTICLE,NaN,NaN,NaN,NaN,NaN,"Polycentric Neoliberalism: Circulation, Transl...",NaN,item_3705148,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2331,CONTRIBUTION_TO_COLLECTED_EDITION,NaN,NaN,NaN,NaN,NaN,RIPE Series in Global Political Economy,NaN,item_3484993,NaN,NaN
2332,CONTRIBUTION_TO_HANDBOOK,Maccarrone,Vincenzo,Vincenzo Maccarrone,NaN,"University College Dublin, Ireland",The Routledge Handbook of the Gig Economy,2022.0,item_3458140,NaN,NaN
2333,CONTRIBUTION_TO_HANDBOOK,Tassinari,Arianna,Arianna Tassinari,CONE,"Politische Ökonomie, MPI for the Study of Soci...",NaN,NaN,item_3458140,female,Senior Researcher
2334,CONTRIBUTION_TO_HANDBOOK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,item_3458140,NaN,NaN


In [7]:
# Datumsspalte bereinigen, da durch Excel-Export verändert und Werte auffüllen
df_glp_cleaned['Datum'] = df_glp_cleaned['Datum'].astype(str).str.extract(r'(\d{4})')
df_glp_cleaned['Datum'] = df_glp_cleaned['Datum'].ffill() 

# Spalteninhalt wieder als Zahl definieren
df_glp_cleaned['Datum'] = df_glp_cleaned['Datum'].astype(int)

# Blick auf die enthaltenen Daten
df_glp_cleaned['Datum'].unique()

array([2026, 2025, 2024, 2023, 2017, 2022, 2020, 2021])

In [8]:
# Alle Zeilen löschen, in denen kein Name vorhanden ist (das löscht auch zweite Affiliations, aber die brauchen wir hier ja nicht!)
before = len(df_glp_cleaned)
mask = df_glp_cleaned['Name'].notna() & (df_glp_cleaned['Name'].astype(str).str.strip() != '')
df_glp_red = df_glp_cleaned[mask].copy()
after = len(df_glp_red)
print(f"Dropped {before - after} rows; remaining {after}")

Dropped 727 rows; remaining 853


# Begrenzung auf bestimmte Jahr(e), hier definieren:

In [9]:
years = [2023, 2024, 2025]   # oder bei einem Einzahljahr so [2025]

df_matching_years = df_glp_red[df_glp_red['Datum'].isin(years)].copy()

In [10]:
df_matching_years

,Genre,Nachname,Vorname,Name,Cone,Affiliation,1101_Quellentitel,Datum,ItemID,Gender,Status
18,ARTICLE,Kullick,Paul Niklas,Paul Niklas Kullick,CONE,International Max Planck Research School on th...,Finance and Society,2025,item_3674340,male,Doctoral Student
22,ARTICLE,Petry,Johannes,Johannes Petry,NaN,"Institute for Political Science, Goethe Univer...",NaN,2025,item_3674340,NaN,NaN
24,ARTICLE,Cigna,Luca,Luca Cigna,CONE,"Politische Ökonomie, MPI for the Study of Soci...",Regulation & Governance,2025,item_3674550,male,Postdoc
28,ARTICLE,Di Carlo,Donato,Donato Di Carlo,NaN,London School of Economics and Political Scien...,NaN,2025,item_3674550,male,Senior Researcher
29,ARTICLE,Durazzi,Niccolò,Niccolò Durazzi,NaN,"University of Modena & Reggio Emilia, Italy",NaN,2025,item_3674550,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2307,COLLECTED_EDITION,Braun,Benjamin,Benjamin Braun,CONE,"Wirtschaftssoziologie, MPI for the Study of So...",RIPE Series in Global Political Economy,2023,item_3484988,male,Senior Researcher
2310,COLLECTED_EDITION,Koddenbrock,Kai,Kai Koddenbrock,CONE,Projekte von Gastwissenschaftlern und Postdoc-...,NaN,2023,item_3484988,male,Other
2314,BOOK,Marktanner,Alina,Alina Marktanner,CONE,International Max Planck Research School on th...,Quellen und Darstellungen zur Zeitgeschichte,2023,item_3480499,female,Other
2327,CONTRIBUTION_TO_COLLECTED_EDITION,Braun,Benjamin,Benjamin Braun,CONE,"Wirtschaftssoziologie, MPI for the Study of So...",Capital Claims: Power and Global Finance,2023,item_3484993,male,Senior Researcher


In [11]:
# Count unique ItemIDs in common dataframes (adjust dataframe name if needed)
def count_unique_itemids(df, col='ItemID'):
    return df[col].nunique()

print("Datensätze in der Datei nach Bereinigung:", count_unique_itemids(df_glp_cleaned))

print("Datensätze in der Datei nach entfernen der Leerzeilen:", count_unique_itemids(df_glp_red))

print("Datensätze in der Datei nach Reduktion der Jahre:", count_unique_itemids(df_matching_years))


Datensätze in der Datei nach Bereinigung: 447
Datensätze in der Datei nach entfernen der Leerzeilen: 447
Datensätze in der Datei nach Reduktion der Jahre: 399


In [12]:
# Hinzufügen eines Zeilenzählers für die einzelnen IDs 
df_matching_years['Item_seq'] = df_matching_years.groupby('ItemID').cumcount() + 1 

In [13]:
# Ausgangsdatei für alles weitere, Name wieder zurück auf df 
df = df_matching_years

<hr>

## Kleiner Exkurs zur Ermittlung von Erstautor*innen des Instituts 

# Bitte den Suchstring für eigenes Institut verändern  


(**Bitte beachten:** Das ist nur zur Orientierung, da es angewandt wird auf alle Daten der ausgwählten Jahre, aber noch keine Auswahl des Personenstatus, die noch kommt!)

In [49]:
# Zählung wie viele Einträge es gibt für Personen mit unserer Affiliation und ItemSeq 1, also Er
df_aff = df[df['Affiliation'].str.contains('MPI for the Study of Societies', na=False, case=False)]

# Zähler der individuellen ItemIDs in den ausgewählten Daten
all_itemids = df_aff['ItemID'].unique()

# Suche nach Zeilen mit Item_seq 1
itemids_with_seq1 = df_aff[df_aff['Item_seq'] == 1]['ItemID'].unique()

# Ausrechnen der Einträge ohne Item_seq 1
itemids_without_seq1 = set(all_itemids) - set(itemids_with_seq1)

# Und zählen dieser Einträge
count_without_seq1 = len(itemids_without_seq1)

print(f"Gesamtzahl der Datensätze: {len(all_itemids)}")             # Hier sind weniger als für den ausgewählten Zeitraum, was mit kleinen Abweichungen in Affiliations zu tun hat (z.b. auch durch Änderung des Status im Zugehörigkeitszeitraum)
print(f"Datensätze mit MPIfG Erstautor*innen: {len(itemids_with_seq1)}")
print(f"Datensätze wo MPIfG Zweit- oder weiterer Autor*in ist: {count_without_seq1}")

Gesamtzahl der Datensätze: 397
Datensätze mit MPIfG Erstautor*innen: 342
Datensätze wo MPIfG Zweit- oder weiterer Autor*in ist: 55


In [50]:
# Ansicht der unterschiedlichen vorkommenden Genres
df['Genre'].unique()   

array(['ARTICLE', 'CONTRIBUTION_TO_COLLECTED_EDITION', 'BOOK_ITEM',
       'DATA_PUBLICATION', 'EDITORIAL', 'COLLECTED_EDITION', 'THESIS',
       'PREPRINT', 'CONTRIBUTION_TO_HANDBOOK', 'ISSUE', 'BOOK_REVIEW',
       'PAPER', 'CONTRIBUTION_TO_ENCYCLOPEDIA', 'BOOK', 'BLOG_POST',
       'HANDBOOK'], dtype=object)

<hr>

# Falls Status nicht relevant ist, folgen zwei Zeilen und alles für Abzug Statustabelle auskommentieren

In [51]:
# Schritt der Reduzierung auf bestimmte Positionen im Status - da wir beschlossen haben Emeriti, Associates, Visiting und Other rauszulassen

desired_statuses = ['Doctoral Student','Postdoc', 'Senior Researcher','Research Group Leader','Director']
df_final = df[df['Status'].isin(desired_statuses)]

In [52]:
df = df_final # Umbenennung für den Fall, dass Status nicht relevant ist und für den Durchlauf für das weitere Skript

In [53]:
# Alle Auswahlkriterien sind eingeflossen, jetzt wird die Datei exportiert, falls sie anderweitig genutzt werden soll.
df_final.to_excel('Daten/Rohdaten_GLP.xlsx', index=False)


<hr>

# Abzug für Statustabelle

In [57]:
df_status = df[['Name', 'Gender','Status', 'Genre', 'Datum', 'ItemID']]
df_status

,Name,Gender,Status,Genre,Datum,ItemID
18,Paul Niklas Kullick,male,Doctoral Student,ARTICLE,2025,item_3674340
24,Luca Cigna,male,Postdoc,ARTICLE,2025,item_3674550
28,Donato Di Carlo,male,Senior Researcher,ARTICLE,2025,item_3674550
41,Stephan Gruber,male,Doctoral Student,BOOK_ITEM,2024,item_3598030
49,Benjamin Braun,male,Senior Researcher,DATA_PUBLICATION,2024,item_3580346
...,...,...,...,...,...,...
2280,Björn Bremer,male,Senior Researcher,CONTRIBUTION_TO_HANDBOOK,2023,item_3506542
2290,Lucio Baccaro,male,Director,ARTICLE,2023,item_3502391
2301,Sebastian Kohl,male,Senior Researcher,BLOG_POST,2023,item_3493276
2307,Benjamin Braun,male,Senior Researcher,COLLECTED_EDITION,2023,item_3484988


In [58]:
# 1. Zähle Publikationen pro Name
df_status['Number of Publications'] = df_status.groupby('Name')['ItemID'].transform('count')

# 2. Zusammenfassung Publikationen pro Status
summary_per_status = (
    df_status.groupby('Status', as_index=False)
    .agg(
        Publikationen_gesamt=('ItemID', 'count'),
        Female_Authors=('Gender', lambda x: (x == 'female').sum()),
        Male_Authors=('Gender', lambda x: (x == 'male').sum()),
        Divers_Authors=('Gender', lambda x: (x == 'divers').sum())
    )
    .sort_values('Status')
)

# 3. Berechne Prozentanteile für Publikationen pro Status
summary_per_status['Anteil female'] = (
    (summary_per_status['Female_Authors'] / summary_per_status['Publikationen_gesamt']) * 100
).round(0).astype(int)

summary_per_status['Anteil male'] = (
    (summary_per_status['Male_Authors'] / summary_per_status['Publikationen_gesamt']) * 100
).round(0).astype(int)

summary_per_status['Anteil divers'] = (
    (summary_per_status['Divers_Authors'] / summary_per_status['Publikationen_gesamt']) * 100
).round(0).astype(int)

summary_per_status = summary_per_status.round(0) # Zahlen runden

# 4. Zusammenfassung Autorenzahl pro Status
person_gender = df_status.drop_duplicates(subset=['Name', 'Status'])  # Nur ein Eintrag pro Person pro Status


summary_per_status_aut = (
    person_gender.groupby('Status', as_index=False)
    .agg(
        Alle_Personen=('Name', 'nunique'),
        Unique_Female_Authors=('Gender', lambda x: (x == 'female').sum()),
        Unique_Male_Authors=('Gender', lambda x: (x == 'male').sum()),
        Unique_Divers_Authors=('Gender', lambda x: (x == 'divers').sum())
    )
    .sort_values('Status')
)

# 5 Berechnung Prozentanteile für Personen pro Status
summary_per_status_aut['Female_Percent'] = (
    (summary_per_status_aut['Unique_Female_Authors'] / summary_per_status_aut['Alle_Personen']) * 100
).round(0).astype(int)

summary_per_status_aut['Male_Percent'] = (
    (summary_per_status_aut['Unique_Male_Authors'] / summary_per_status_aut['Alle_Personen']) * 100
).round(0).astype(int)

summary_per_status_aut['Divers_Percent'] = (
    (summary_per_status_aut['Unique_Divers_Authors'] / summary_per_status_aut['Alle_Personen']) * 100
).round(0).astype(int)



# 6. Zusammenfügen
overview = summary_per_status_aut.merge(summary_per_status, on='Status', how='left')

#overview

C:\Users\cgmol\AppData\Local\Temp\ipykernel_56968\2311525050.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_status['Number of Publications'] = df_status.groupby('Name')['ItemID'].transform('count')


In [ ]:
# Umsortierung und Vereinfachung der Namen, sowie Export zu Excel

overview_sorted = overview[['Status',
                           'Alle_Personen',
                           'Unique_Female_Authors',
                           'Female_Percent',
                           'Unique_Male_Authors',
                           'Male_Percent',
                           'Unique_Divers_Authors',
                           'Divers_Percent',
                           'Publikationen_gesamt',
                           'Female_Authors',
                           'Anteil female',
                           'Male_Authors',
                           'Anteil male',
                           'Divers_Authors',
                           'Anteil divers']]



status_final = overview_sorted.rename(columns={    #Vereinfachung der Spaltennamen
    'Alle_Personen':'Autoren gesamt',
    'Unique_Female_Authors' : 'davon weiblich',
    'Female_Percent':'%-Anteil',
    'Unique_Male_Authors':'davon männlich',
    'Male_Percent': '%-Anteil',
    'Unique_Divers_Authors':'davon divers',
    'Divers_Percent':'%-Anteil',
    'Publikationen_gesamt':'Publikationen gesamt',
    'Female_Authors': 'davon weibl. Autoren',
    'Anteil female':'%-Anteil',
    'Male_Authors':'davon männl. Autoren',
    'Anteil male':'%-Anteil',
    'Divers_Authors':'davon diverse Autoren',
    'Anteil divers':'%-Anteil'}) 

status_final.to_excel('Results/GLP_Autorenschaft_Gender_Status.xlsx', index=False)
status_final

,Status,Autoren gesamt,davon weiblich,%-Anteil,davon männlich,%-Anteil,davon divers,%-Anteil,Publikationen gesamt,davon weibl. Autoren,%-Anteil,davon männl. Autoren,%-Anteil,davon diverse Autoren,%-Anteil
0,Director,2,0,0,2,100,0,0,31,0,0,31,100,0,0
1,Doctoral Student,35,17,49,18,51,0,0,95,50,53,45,47,0,0
2,Postdoc,12,6,50,6,50,0,0,20,11,55,9,45,0,0
3,Research Group Leader,4,2,50,2,50,0,0,49,13,27,36,73,0,0
4,Senior Researcher,17,6,35,11,65,0,0,155,45,29,110,71,0,0


<hr>

## Hier beginnt die Aufbereitung der Gendertabelle

# ggf. Genre-Zuordnungen für eigenes Institut anpassen

In [14]:
# 📌 Centralized Genre Definitions
genre_config = {
    'book': {
        'genres': ['BOOK'],
        'description': 'Books'
    },
    'article': {
        'genres': ['ARTICLE', 'BOOK_REVIEW', 'EDITORIAL'],
        'description': 'Journal Articles'
    },
    'book_contributions': {
        'genres': [
            'CONTRIBUTION_TO_COLLECTED_EDITION',
            'CONTRIBUTION_TO_ENCYCLOPEDIA',
            'CONTRIBUTION_TO_HANDBOOK',
            'BOOK_ITEM'
        ],
        'description': 'Book Contributions'
    },
    'data': {
        'genres': ['DATA_PUBLICATION'],
        'description': 'Data Publications'
    },
    'thesis': {
        'genres': ['THESIS'],
        'description': 'Theses'
    },
    'DP': {
        'genres': ['PAPER'],
        'description': 'MPIfG Discussion Papers'
    },
    'others': {
        'genres': ['PAPER'],
        'description': 'Others'
    }
}

# Institutsname Anpassen

In [16]:
# 📌 Define the exact affiliation string, so only MPIfG Members are taken into account
MPI_AFFILIATION_PHRASE = "MPI for the Study of Societies"

Aufgrund von Datenproblemen hier starke Debugging-Elemente einbauen lassen

In [17]:
for genre_name, config in genre_config.items():
    genre_list = config['genres']
    
    for year in years:
        print(f"\n{'='*60}")
        print(f"Processing: {genre_name.upper()} ({config['description']}) | Year: {year}")
        print(f"{'='*60}")

        # Normalize inputs
        if isinstance(genre_list, str):
            genre_list = [genre_list]
        if isinstance(year, (int,)):
            year_list = [str(year)]
        elif isinstance(year, str):
            year_list = [year]
        else:
            year_list = [str(year)]

        # 📌 Base filtering: Genre, Year, AND Affiliation (applies to ALL genres)
        df_genre = df[df['Genre'].isin(genre_list)].copy()
        print(f"  Debug: df_genre shape after genre filter: {df_genre.shape}")
        print(f"  Debug: year_list: {year_list}")
        if 'Datum' in df_genre.columns:
            print(f"  Debug: 'Datum' column dtype: {df_genre['Datum'].dtype}")
            print(f"  Debug: Unique values in 'Datum' column: {df_genre['Datum'].unique()}")

            # Convert 'Datum' to year string for comparison
            if pd.api.types.is_datetime64_any_dtype(df_genre['Datum']):
                df_genre['Datum_Year_Str'] = df_genre['Datum'].dt.year.astype(str)
            elif pd.api.types.is_numeric_dtype(df_genre['Datum']):
                df_genre['Datum_Year_Str'] = df_genre['Datum'].astype(str)
            else: # Assume it's a string-like object, try to extract first 4 chars
                df_genre['Datum_Year_Str'] = df_genre['Datum'].astype(str).str[:4]
            
            df_genre_year = df_genre[df_genre['Datum_Year_Str'].isin(year_list)].copy()
            df_genre = df_genre.drop(columns=['Datum_Year_Str'])
        else:
            print("  Debug: 'Datum' column not found in df_genre. Creating empty df_genre_year.")
            df_genre_year = pd.DataFrame()

        print(f"  Debug: df_genre_year shape after year filter (before affiliation): {df_genre_year.shape}")
        if 'Affiliation' in df_genre_year.columns:
            print(f"  Debug: Unique values in 'Affiliation' column (before filter): {df_genre_year['Affiliation'].unique()}")
        else:
            print("  Debug: 'Affiliation' column not found in df_genre_year.")
        print(f"  Debug: MPI_AFFILIATION_PHRASE: {MPI_AFFILIATION_PHRASE}")

        if not df_genre_year.empty and 'Affiliation' in df_genre_year.columns:
            mpi_filter = df_genre_year['Affiliation'].str.contains(MPI_AFFILIATION_PHRASE, na=False, case=False)
            df_genre_year = df_genre_year[mpi_filter].copy()
        else:
            # If df_genre_year is already empty or no 'Affiliation' column, no need to filter further
            pass
        print(f"  ✅ Filtered by affiliation: {df_genre_year.shape[0]} entries with '{MPI_AFFILIATION_PHRASE}'")

        # 🔍 Conditional: Only for DP → filter for MPIfG Discussion Paper
        if genre_name == "DP":
            if not df_genre_year.empty and '1101_Quellentitel' in df_genre_year.columns:
                dp_filter = df_genre_year['1101_Quellentitel'].str.contains('MPIfG Discussion Paper', na=False)
                df_genre_year = df_genre_year[dp_filter].copy()
            print(f"  ✅ Filtered DP: {df_genre_year.shape[0]} entries with 'MPIfG Discussion Paper'")

        # 🔍 Conditional: Only for 'other' → add non-MPIfG PAPER entries
        elif genre_name == "other":
            if not df_genre_year.empty and '1101_Quellentitel' in df_genre_year.columns:
                paper_filter = (
                    (df_genre_year['Genre'] == 'PAPER') &
                    (~df_genre_year['1101_Quellentitel'].str.contains('MPIfG Discussion Paper', na=False))
                )
                additional_papers = df_genre_year[paper_filter].copy()
                if not additional_papers.empty:
                    print(f"  ✅ Added {additional_papers.shape[0]} non-MPIfG PAPER entries")
                    df_genre_year = pd.concat([df_genre_year, additional_papers], ignore_index=True)
                else:
                    print("  ❌ No non-MPIfG PAPER entries to add")
            else:
                 print("  ❌ No non-MPIfG PAPER entries to add (df_genre_year empty or missing '1101_Quellentitel')")

        # ✅ Create variable name
        var_name = f"df_genre_{year}_{genre_name}"
        globals()[var_name] = df_genre_year.copy()

        # ✅ Print summary
        print(f"  Final shape: {df_genre_year.shape}")
        print(f"  Saved to: {var_name}")


Processing: BOOK (Books) | Year: 2023
  Debug: df_genre shape after genre filter: (28, 12)
  Debug: year_list: ['2023']
  Debug: 'Datum' column dtype: int64
  Debug: Unique values in 'Datum' column: [2024 2025 2023]
  Debug: df_genre_year shape after year filter (before affiliation): (10, 13)
  Debug: Unique values in 'Affiliation' column (before filter): ['Projekte der Emeriti, MPI for the Study of Societies, Max Planck Society'
 nan
 'Projekte von Gastwissenschaftlern und Postdoc-Stipendiaten, MPI for the Study of Societies, Max Planck Society'
 'Politische Ökonomie, MPI for the Study of Societies, Max Planck Society'
 'School of Law, University of Glasgow, UK'
 'Soziologie öffentlicher Finanzen und Schulden, MPI for the Study of Societies, Max Planck Society'
 'International Max Planck Research School on the Social and Political Constitution of the Economy, MPI for the Study of Societies, Max Planck Society']
  Debug: MPI_AFFILIATION_PHRASE: MPI for the Study of Societies
  ✅ Filte

In [20]:
# Zusammenfügen der einzelnen Auswertungen zu einer Tabelle

# 📌 Step 1: Collect all genre-year DataFrames from globals()
matched_dfs = {
    name: value for name, value in globals().items()
    if name.startswith("df_genre_") and isinstance(value, pd.DataFrame)
}

print(f"✅ Found {len(matched_dfs)} genre-year DataFrames.")

# 📌 Step 2: Prepare the summary list
summary_data = []

# 📌 Step 3: Loop through each DataFrame and extract summary stats
for var_name, df in matched_dfs.items():
    print(f"\n🔍 Processing: {var_name}")

    # Extract year and genre from variable name: e.g., df_genre_2025_book
    parts = var_name.replace("df_genre_", "").split("_")
    if len(parts) < 2:
        print(f"  ❌ Skipping malformed name: {var_name}")
        continue
    year = parts[0]
    genre_name = "_".join(parts[1:])

    # Get description from config
    description = genre_config.get(genre_name, {}).get('description', 'Unknown')

    # 🔍 ✅ Step 3.1: Check if 'Gender' column exists and is cleaned
    if 'Gender' not in df.columns:
        print(f"  ❌ No 'Gender' column in {var_name} — skipping.")
        continue

    # 🔍 ✅ Step 3.2: Ensure Gender is cleaned (e.g., 'Male', 'Female', 'Divers', 'Missing')
    # If not, clean it now
    if not df['Gender'].isin(['Male', 'Female', 'Divers', 'Missing']).all():
        print(f"  ⚠️ Cleaning Gender column in {var_name}...")
        df_clean = df.copy()
        df_clean['Gender'] = df_clean['Gender'].replace({
            'M': 'Male',
            'F': 'Female',
            'male': 'Male',
            'female': 'Female',
            'Diverse': 'Divers',
            'Other': 'Divers',
            'Non-binary': 'Divers',
            'Non Binary': 'Divers',
            '': 'Missing',
            np.nan: 'Missing'
        })
    else:
        df_clean = df  # Already cleaned

    # 🔍 ✅ Step 3.3: Count gender values
    gender_counts = df_clean['Gender'].value_counts().to_dict()
    male = gender_counts.get('Male', 0)
    female = gender_counts.get('Female', 0)
    divers = gender_counts.get('Divers', 0)
    missing = gender_counts.get('Missing', 0)

    # 🔍 ✅ Step 3.4: Total count
    total = len(df_clean)

    # 🔍 ✅ Step 3.5: Calculate percentages (safe division)
    male_pct = (male / total) * 100 if total > 0 else 0
    female_pct = (female / total) * 100 if total > 0 else 0
    divers_pct = (divers / total) * 100 if total > 0 else 0

    # 🔍 ✅ Step 3.6: Print debug info
    print(f"  ✅ Total: {total}, Male: {male}, Female: {female}, Divers: {divers}, Missing: {missing}")
    print(f"  ✅ Percentages: {male_pct:.1f}%, {female_pct:.1f}%, {divers_pct:.1f}%")

    # Append to summary
    summary_data.append({
        'Year': year,
        'Genre': genre_name,
        'Description': description,
        'Total': total,
        'Male': male,
        'Female': female,
        'Divers': divers,
        'Missing': missing,
        'Male %': round(male_pct, 1),
        'Female %': round(female_pct, 1),
        'Divers %': round(divers_pct, 1)
    })

# 📌 Step 4: Create the master summary DataFrame
master_summary = pd.DataFrame(summary_data)

# 📌 Step 5: Sort and reset index
master_summary = master_summary.sort_values(['Year', 'Genre']).reset_index(drop=True)

# 📌 Step 6: Add yearly totals (with proper rounding)
yearly_totals = master_summary.groupby('Year', as_index=False)[['Total', 'Male', 'Female', 'Divers']].sum()
yearly_totals['Genre'] = 'Total'
yearly_totals['Description'] = 'Yearly Total'
yearly_totals['Missing'] = 0
yearly_totals['Male %'] = (yearly_totals['Male'] / yearly_totals['Total']) * 100
yearly_totals['Female %'] = (yearly_totals['Female'] / yearly_totals['Total']) * 100
yearly_totals['Divers %'] = (yearly_totals['Divers'] / yearly_totals['Total']) * 100
yearly_totals = yearly_totals.round(1)

# Append totals
master_summary = pd.concat([master_summary, yearly_totals], ignore_index=True)


✅ Found 22 genre-year DataFrames.

🔍 Processing: df_genre_year
  ❌ Skipping malformed name: df_genre_year

🔍 Processing: df_genre_2023_book
  ⚠️ Cleaning Gender column in df_genre_2023_book...
  ✅ Total: 7, Male: 6, Female: 1, Divers: 0, Missing: 0
  ✅ Percentages: 85.7%, 14.3%, 0.0%

🔍 Processing: df_genre_2024_book
  ⚠️ Cleaning Gender column in df_genre_2024_book...
  ✅ Total: 5, Male: 4, Female: 1, Divers: 0, Missing: 0
  ✅ Percentages: 80.0%, 20.0%, 0.0%

🔍 Processing: df_genre_2025_book
  ⚠️ Cleaning Gender column in df_genre_2025_book...
  ✅ Total: 6, Male: 4, Female: 1, Divers: 0, Missing: 0
  ✅ Percentages: 66.7%, 16.7%, 0.0%

🔍 Processing: df_genre_2023_article
  ⚠️ Cleaning Gender column in df_genre_2023_article...
  ✅ Total: 66, Male: 44, Female: 21, Divers: 0, Missing: 0
  ✅ Percentages: 66.7%, 31.8%, 0.0%

🔍 Processing: df_genre_2024_article
  ⚠️ Cleaning Gender column in df_genre_2024_article...
  ✅ Total: 79, Male: 53, Female: 26, Divers: 0, Missing: 0
  ✅ Percentages: 

In [ ]:
# master_summary  #wenn man es sich ansehen möchte, aktivieren

Achtung: Spalte "Total" zählt alle MPIfG-Autor*innen, nicht die Anzahl der Titel, aber ist ja logisch, um die Geschlechteranteile auszurechnen

In [ ]:
final_summary = master_summary[['Year', 'Description', 'Male', 'Female', 'Divers', 'Female %']] # auf relevante Spalten reduzieren

final_summary.to_excel("Results/GLP_summary_nach_Genre.xlsx", index=False)
final_summary

# das kann dann in Excel noch etwas in Form gebracht werden für die Jahre

,Year,Description,Male,Female,Divers,Female %
0,2023,MPIfG Discussion Papers,3,3,0,50.0
1,2023,Journal Articles,44,21,0,31.8
2,2023,Books,6,1,0,14.3
3,2023,Book Contributions,24,2,0,7.7
4,2023,Data Publications,3,5,0,62.5
5,2023,Others,11,5,0,31.2
6,2023,Theses,4,2,0,33.3
7,2024,MPIfG Discussion Papers,8,1,0,11.1
8,2024,Journal Articles,53,26,0,32.9
9,2024,Books,4,1,0,20.0
